# CSET485 – AI and Society | Assignment 1
**Student:** Devansh Srivastava  
**Enrollment number:** S24CSEU1646  
**Assigned dataset:** Adult Census Income (last digit = 6)  
**Random seed:** 646  

This notebook is the reproducible working file for the assignment. Run the cells from top to bottom. All numerical claims in the final report must come from the outputs generated here.

## 0. Setup

Install the required packages once in your environment. In Jupyter, you can run the command below if a package is missing.

In [ ]:
# If needed, uncomment and run:
# %pip install ucimlrepo pandas numpy scikit-learn statsmodels matplotlib seaborn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from IPython.display import display

RANDOM_STATE = 646
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries loaded. Random seed:", RANDOM_STATE)

## 1. Load the assigned dataset

The assignment assigns Adult Census Income to enrollment numbers ending in 6, 7, 8, or 9. The UCI repository describes this dataset as a census-based task for predicting whether annual income exceeds $50,000. It contains demographic and employment-related attributes.

In [ ]:
# Fetch the official UCI Adult dataset (dataset ID 2)
adult = fetch_ucirepo(id=2)

X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)
display(X_raw.head())
display(y_raw.head())

## 2. Clean column values and create the target

The target is encoded as `1` for income greater than $50K and `0` for income of $50K or less. The Adult data uses `?` for some missing categorical values; these are converted to actual missing values and imputed later inside the preprocessing pipeline.

In [ ]:
# Normalize column names for convenient Python use
X = X_raw.copy()
X.columns = [str(c).strip().replace("-", "_") for c in X.columns]

# Convert whitespace-only values and '?' markers to missing values
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(str).str.strip()
        X[col] = X[col].replace({"?": np.nan, "": np.nan, "nan": np.nan})

# Normalize target text (some versions may include a trailing period)
target_col = y_raw.columns[0]
y_text = y_raw[target_col].astype(str).str.strip().str.replace(".", "", regex=False)
y = (y_text == ">50K").astype(int)
y.name = "income_binary"

print("Target counts (0 = <=50K, 1 = >50K):")
display(y.value_counts().rename_axis("income_binary").to_frame("count"))
print("\nMissing values by feature:")
display(X.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 3. Document preprocessing decisions

- **Missing values:** replace `?` with missing values; use the most frequent category for categorical columns and the median for numeric columns.
- **Categorical encoding:** one-hot encode categorical features; ignore unseen categories at test time.
- **Feature scaling:** standardize numeric features. Scaling is fitted only on the training set through a pipeline, preventing test-set leakage.
- **Outliers:** inspect numeric distributions and IQR-based counts. Do not delete observations automatically; retain genuine extreme income-related values and let the models handle them. Any decision to cap/remove observations must be justified and documented.

In [ ]:
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# IQR-based outlier inspection (counts only; no rows removed at this stage)
outlier_summary = []
for col in numeric_cols:
    q1 = X[col].quantile(0.25)
    q3 = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((X[col] < lower) | (X[col] > upper)).sum()
    outlier_summary.append([col, lower, upper, int(count), round(100 * count / len(X), 2)])

outlier_summary = pd.DataFrame(
    outlier_summary,
    columns=["feature", "IQR_lower_bound", "IQR_upper_bound", "outlier_count", "outlier_percent"]
)
display(outlier_summary)

## 4. Train-test split

Use the required 70% training / 30% testing split and `random_state=646` (the last three digits of the enrollment number).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training proportion:", round(len(X_train) / len(X), 3))
print("Testing proportion:", round(len(X_test) / len(X), 3))
print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 5. Build reusable preprocessing pipelines

The same preprocessing design will be used for both models so that the comparison is fair. Imputation, encoding, and scaling are fitted on the training data only.

In [ ]:
numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocess, numeric_cols),
    ("categorical", categorical_preprocess, categorical_cols)
])

# Fit once on training data; transform both training and test sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 6. Part 1 – Build Model A and Model B

For this dataset:
- **Model A:** Linear Regression (OLS) predicts the binary income target as a continuous score.
- **Model B:** Logistic Regression predicts the probability that income is greater than $50K.

The assignment later compares Model A using its median predicted-value threshold against Model B using a probability threshold of 0.5.

In [ ]:
# Model A: Linear Regression / OLS-style prediction
model_a = LinearRegression()
model_a.fit(X_train_processed, y_train)
pred_a = model_a.predict(X_test_processed)

# Model B: Logistic Regression
model_b = LogisticRegression(max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)
model_b.fit(X_train_processed, y_train)
prob_b = model_b.predict_proba(X_test_processed)[:, 1]

print("Model A predictions (first 10):", np.round(pred_a[:10], 4))
print("Model B probabilities (first 10):", np.round(prob_b[:10], 4))

## 7. Part 1 – Compare the two thresholding approaches

In [ ]:
median_a = float(np.median(pred_a))
class_a = (pred_a >= median_a).astype(int)
class_b = (prob_b >= 0.50).astype(int)

cm_agreement = confusion_matrix(class_a, class_b, labels=[0, 1])
mismatch_count = int(np.sum(class_a != class_b))
mismatch_percent = 100 * mismatch_count / len(class_a)

print(f"Median of Model A predictions: {median_a:.6f}")
print("\nAgreement matrix (rows = Model A class, columns = Model B class):")
display(pd.DataFrame(
    cm_agreement,
    index=["Model A: low (0)", "Model A: high (1)"],
    columns=["Model B: low (0)", "Model B: high (1)"]
))
print(f"Mismatched cases: {mismatch_count} / {len(class_a)} ({mismatch_percent:.2f}%)")

## 8. Initial results to record

This section captures the reproducible dataset and model details. Do not write final descriptive answers until Parts 2.1–2.4 have been run and their actual results are available.

In [ ]:
results_summary = pd.DataFrame({
    "item": [
        "Dataset rows",
        "Dataset feature columns",
        "Training rows",
        "Testing rows",
        "Random seed",
        "Model A median prediction",
        "Part 1 mismatched cases",
        "Part 1 mismatch percentage"
    ],
    "value": [
        len(X),
        X.shape[1],
        len(X_train),
        len(X_test),
        RANDOM_STATE,
        median_a,
        mismatch_count,
        mismatch_percent
    ]
})
display(results_summary)

---
**Dataset citation:** Becker, B. & Kohavi, R. (1996). *Adult* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20

**Next stage:** add Part 2 stress tests (Simpson's Paradox, omitted-variable bias, adversarial subset, and fairness trade-offs), then generate the evidence-based written answers and final PDF.